In [ ]:
import os, json, boto3, sagemaker
# Base-vs-student tool-use accuracy for the SeqKD distillation recipe. Tool-use ASR is the headline
# metric; MMLU + non-thinking IFEval guard against regression. Each axis runs on the compute that fits
# it: MMLU on a managed serverless benchmark; IFEval + ASR via a driver here against a real-time endpoint.

## 0.0 Update the system prompt for your task and tools

In [ ]:
system_prompt = (
    "You are a shopping assistant. Use the available tools to satisfy the request: "
    "search the catalog, inspect candidates, verify the constraints, then recommend."
)
# Held-out ShoppingBench eval problems (query + tools + gold product/criteria).
# eval_dataset = load_dataset(...)  # your held-out split

## 1.0 Deploy the merged model to a real-time endpoint (`ModelBuilder`)

`ModelBuilder` deploys the merged model straight from S3. Name the LMI image `763104351884.dkr.ecr.<region>.amazonaws.com/djl-inference:0.36.0-lmi25.0.0-cu130` and `model_server=DJL_SERVING` **explicitly** — a raw-S3 model otherwise defaults to TorchServe (`Cannot detect required model or inference spec`). That image bundles **vLLM 0.20.1**, which serves the Qwen3.5 linear-attn/conv1d hybrid, so **no custom inference image is needed** (validated: Qwen3.5-4B served on the stock image). (You can also point `model=` at the training job, but a LoRA run saves an *adapter* — merge it first, as in the training notebook's merge step.)

In [ ]:
from sagemaker.serve import ModelBuilder, ModelServer

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
bucket = sagemaker.Session(boto3.Session(region_name=region)).default_bucket()
MERGED_S3 = f"s3://{bucket}/distillation/seqkd-shopping/qwen3.5-4b/merged/r64/"   # from the merge step
ENDPOINT_NAME = "seqkd-qwen35-4b-shop-r64"
LMI_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.36.0-lmi25.0.0-cu130"  # vLLM 0.20.1

# A raw S3 model otherwise defaults to the TorchServe path ("Cannot detect required model or inference
# spec"), so name the LMI image + DJL model server explicitly. env_vars configure the vLLM engine;
# OPTION_TRUST_REMOTE_CODE is required for the Qwen3.5 hybrid, TP=1 fits the 4B on a single GPU.
mb = ModelBuilder(
    s3_model_data_url={"S3DataSource": {"S3Uri": MERGED_S3,
                                        "S3DataType": "S3Prefix", "CompressionType": "None"}},
    role_arn=role,
    image_uri=LMI_IMAGE,
    model_server=ModelServer.DJL_SERVING,
    instance_type="ml.g5.2xlarge",
    env_vars={"OPTION_ROLLING_BATCH": "vllm", "OPTION_TRUST_REMOTE_CODE": "true",
              "OPTION_TENSOR_PARALLEL_DEGREE": "1", "OPTION_MAX_MODEL_LEN": "8192",
              "HF_MODEL_ID": "/opt/ml/model"},
)
# Alt - deploy straight from the training job (merge first; LoRA runs save an adapter):
# from sagemaker.core.resources import TrainingJob
# mb = ModelBuilder(model=TrainingJob.get(training_job_name="seqkd-shop-lora-r64-4k-p5-20260930110555"),
#                   role_arn=role, image_uri=LMI_IMAGE, model_server=ModelServer.DJL_SERVING)

mb.model_name = ENDPOINT_NAME
mb.build()                                     # -> djl-inference:0.36.0-lmi25.0.0-cu130 (vLLM 0.20.1)
mb.deploy(instance_type="ml.g5.2xlarge", endpoint_name=ENDPOINT_NAME, initial_instance_count=1)
print("endpoint:", ENDPOINT_NAME)

## 2.0 Tool-use ASR + IFEval — driver → endpoint

These run in **this notebook kernel** (CPU-light — orchestration + scoring only) and send requests to the endpoint, whose GPU does the generation. SageMaker endpoints are invoked via `InvokeEndpoint`, not an OpenAI URL, so **LiteLLM** bridges the gap — either call `litellm.completion(model=f"sagemaker/{ENDPOINT_NAME}", ...)` directly, or run `litellm --model sagemaker/<endpoint>` as a proxy that exposes an OpenAI `base_url` for `lm_eval` / the rollout to point at.

**Thinking-off (important):** on a managed endpoint you can't pass `--chat-template nonthink_template.jinja` like the self-hosted path, so force non-thinking **per request** with `extra_body={"enable_thinking": false}` (mirrors the rollout config). IFEval floors (~28%) if thinking is left on.

**ASR** reuses the same ReAct rollout as `scripts/shopping_asr_eval.sh` / `run_rollout.py`, only with `base_url` pointed at the LiteLLM shim instead of a local vLLM. It still needs the **ShoppingBench repo + search server (:5631) + `JAVA_HOME`** for tool results and orm.py scoring.

In [ ]:
# Invoke schema: this endpoint serves the TGI `{"inputs":..., "parameters":...}` schema (returns {"generated_text"}).
# The OpenAI `{"messages":...}` schema returns 424 unless the chat route is enabled; LiteLLM's `sagemaker/` provider uses the TGI schema under the hood.
# --- LiteLLM proxy: expose the endpoint as an OpenAI base_url ---
#   pip install "litellm[proxy]"
#   litellm --model sagemaker/{ENDPOINT_NAME} --port 4000 &        # -> http://localhost:4000
#
# IFEval (non-thinking, generative) — point lm_eval at the proxy:
#   lm_eval --model local-chat-completions \
#     --model_args base_url=http://localhost:4000/v1/chat/completions,model=sagemaker/{ENDPOINT_NAME},num_concurrent=8 \
#     --apply_chat_template --gen_kwargs enable_thinking=false --tasks ifeval
#   # reference: prompt-strict ~82.6 (base), held after distillation

# ASR — drive the ShoppingBench ReAct loop against the endpoint (thinking OFF), score with orm.py:
import litellm
def endpoint_chat(messages, tools=None):
    return litellm.completion(model=f"sagemaker/{ENDPOINT_NAME}", messages=messages, tools=tools,
                              temperature=0, extra_body={"enable_thinking": False})
# for ex in eval_dataset: run the tool loop with endpoint_chat() (search server for tool results),
#     then score the final recommendation with orm.py (success rate = rule>=1)
print("ASR reference: base 43.2 -> 62.4 (LoRA r64)")

## 3.0 MMLU — native serverless benchmark (no endpoint)

`BenchMarkEvaluator` runs MMLU as a **managed serverless** pipeline execution: SageMaker provisions the compute, and because it isn't endpoint-based it sidesteps the token-logprob + transport shim that MMLU (a loglikelihood task) would otherwise need. It targets a **registered Model Package**, so register the merged model first (via `ModelBuilder` / `ModelPackage`) and pass its ARN.

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks
Benchmark = get_benchmarks()

MODEL_PKG_ARN = "<arn:aws:sagemaker:...:model-package/...>"   # register the merged model first
mmlu = BenchMarkEvaluator(
    benchmark=Benchmark.MMLU, model=MODEL_PKG_ARN,
    s3_output_path=f"s3://{bucket}/distillation/seqkd-shopping/qwen3.5-4b/eval/mmlu/",
    base_eval_name="seqkd-r64-mmlu",
    evaluate_base_model=True,          # base vs student in one run
)
execution = mmlu.evaluate()
execution.wait(target_status="Succeeded", poll=30, timeout=3600)
execution.show_results()               # reference: base ~0.697, student ~0.697 (held)

## 4.0 Portfolio — base vs student

> **Alternative (self-contained, no endpoint):** `scripts/eval_lmeval.sh <MERGED_DIR> <IFACE> <GPU> <TAG>` runs MMLU + non-thinking IFEval on a single GPU box, and `scripts/shopping_asr_eval.sh` runs ASR — use these when you'd rather serve a local vLLM than stand up an endpoint.

In [ ]:
import pandas as pd
# base-vs-student portfolio (fill from the runs above; reference numbers shown as fallback)
portfolio = {
    "ASR":    {"base": 0.432, "student": 0.624},
    "IFEval": {"base": 0.826, "student": 0.810},
    "MMLU":   {"base": 0.697, "student": 0.697},
}
df = pd.DataFrame(portfolio).T
df["delta"] = df["student"] - df["base"]
try:
    import matplotlib.pyplot as plt
    ax = df[['base', 'student']].plot(kind='bar', figsize=(8, 4))
    ax.set_title('SeqKD distillation: base vs student (Qwen3.5-4B)')
    ax.set_ylabel('score'); plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()
except Exception as ex:
    print('(matplotlib unavailable)', ex)
df

## 5.0 Tear down the endpoint

Delete it when done — a real-time endpoint bills for its instance while it's up.

In [ ]:
# deploy() returns an Endpoint object (no .delete_endpoint), so delete by name via the sagemaker
# client. ALWAYS tear down - an idle endpoint keeps billing (p4de/g5 are not cheap).
import boto3
sm = boto3.client("sagemaker", region_name=region)
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
sm.delete_model(ModelName=ENDPOINT_NAME)
print("deleted endpoint + config + model:", ENDPOINT_NAME)